# OpenShorts on Kaggle (2×T4)

**This notebook is deliberately thin.** Everything that could need fixing lives
in the repo (`kaggle_bootstrap.sh`, `kaggle_smoke_test.py`), so improvements
arrive with a `git pull` in cell 2 — you should not have to re-import this
notebook again.

**Before running — notebook settings (right panel):**

| setting | value |
|---|---|
| Accelerator | **GPU T4 ×2** |
| Internet | **On** (needs a phone-verified account) |

**Add-ons → Secrets** — create each one and **tick its checkbox** for this
notebook (saving alone does not attach it):

| secret | needed for |
|---|---|
| `GITHUB_TOKEN` | cloning the private repo (PAT with Contents:Read) |
| `GEMINI_API_KEY` | clip selection + scene context — **without it nothing is clipped** |
| `GEMINI_API_KEYS` | optional: extra keys, comma-separated; the pool rotates on 429/quota |
| `ASSEMBLYAI_API_KEY` | transcription **with diarization**. Without it, local whisper runs: ~250s slower per job and no diarization, which the framing policy uses |
| `NARRATIVE_GEMINI_API_KEY` | optional: the fallback clip-selection engine, on its own quota |
| `HF_TOKEN` | **write** token — persists clips past the session, and silences the HF rate-limit warning |
| `HF_STORAGE_REPO` | `<user>/openshorts-clips` — the private dataset repo clips are uploaded to |
| `YOUTUBE_COOKIES` | paste a working cookies.txt if downloads get blocked (Kaggle's IP usually is not) |

Unknown/unset secrets are skipped silently, so you can add one later by
creating it in Kaggle — no notebook edit needed.


In [ ]:
# ── Cell 1 — secrets ───────────────────────────────────────────────────────
# The ONLY thing this notebook knows. Everything else lives in the repo.
#
# Names are loaded from Kaggle Secrets when present. To paste a key instead
# (quicker, but it is saved inside the notebook and its version history — a
# public fork or download takes the key with it), put it in PASTED below.

SECRETS = [
    "GITHUB_TOKEN",
    "GEMINI_API_KEY", "GEMINI_API_KEYS", "NARRATIVE_GEMINI_API_KEY",
    "ASSEMBLYAI_API_KEY",
    "HF_TOKEN", "HF_STORAGE_REPO",
    "YOUTUBE_COOKIES",
    "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_BUCKET",
    "UPLOAD_POST_API_KEY", "ELEVENLABS_API_KEY",
]

PASTED = {
    # "GEMINI_API_KEY": "...",
}

import os

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
except Exception:
    _secrets = None

loaded = []
for name in SECRETS:
    value = (PASTED.get(name) or "").strip()
    source = "pasted"
    if not value and _secrets is not None:
        try:
            value, source = _secrets.get_secret(name).strip(), "secret"
        except Exception:
            value = ""
    if value:
        os.environ[name] = value
        loaded.append(name)
        print(f"  ok        {name:<26} ({source}, {len(value)} chars)")
    else:
        print(f"  not set   {name}")

if "GEMINI_API_KEY" not in loaded:
    print("\n  !! GEMINI_API_KEY is missing — no clips will be selected.")
if "ASSEMBLYAI_API_KEY" not in loaded:
    print("  !  No ASSEMBLYAI_API_KEY — local whisper will run: slower, and no")
    print("     diarization, which costs the framing policy an evidence tier.")
if "HF_TOKEN" not in loaded or "HF_STORAGE_REPO" not in loaded:
    print("  !  No HF storage — clips are wiped when this session ends.")


In [ ]:
# ── Cell 2 — get the code (clone the first time, pull after that) ─────────
import os, subprocess

BRANCH = "claude/gemini-vision-clip-picking-bikvuy"
DEST = "/kaggle/working/openshorts"
PRIVATE_REPO = True   # set False if you make the repo public

have_token = bool(os.environ.get("GITHUB_TOKEN"))
if PRIVATE_REPO and not have_token:
    raise SystemExit(
        "GITHUB_TOKEN not loaded.\n"
        "  -> Add-ons > Secrets: TICK THE CHECKBOX next to GITHUB_TOKEN\n"
        "     (saving the secret is not enough; it must be attached to this\n"
        "     notebook), then re-run cell 1 and this one.")

# Never printed: it carries the token.
url = (f"https://{os.environ['GITHUB_TOKEN']}@github.com/foskigr8/openshorts.git"
       if have_token else "https://github.com/foskigr8/openshorts.git")

if os.path.isdir(os.path.join(DEST, ".git")):
    # Idempotent: re-running this cell picks up new commits instead of
    # reporting "already cloned" and leaving you on stale code.
    subprocess.run(["git", "-C", DEST, "remote", "set-url", "origin", url],
                   capture_output=True, text=True)
    r = subprocess.run(["git", "-C", DEST, "pull", "--ff-only", "origin", BRANCH],
                       capture_output=True, text=True)
    print("pull ok" if r.returncode == 0 else
          f"PULL FAILED (continuing on the existing checkout):\n{r.stderr[-400:]}")
else:
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, DEST],
                       capture_output=True, text=True)
    if r.returncode != 0:
        err = r.stderr[-400:]
        hint = ""
        if "could not read Username" in err:
            hint = "\n  -> the token was not applied; see the checkbox note above"
        elif "Authentication failed" in err or "403" in err:
            hint = ("\n  -> token rejected: check it has Contents:Read on"
                    " foskigr8/openshorts and has not expired")
        elif "Remote branch" in err:
            hint = f"\n  -> branch '{BRANCH}' not found on the remote"
        raise SystemExit(f"CLONE FAILED:\n{err}{hint}")
    print("clone ok")

os.chdir(DEST)
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 3 — install, build, serve, tunnel (5-10 min on a cold session) ───
# Ends by printing a public URL. Re-run with SKIP_INSTALL=1 to skip pip/npm.
!bash kaggle_bootstrap.sh


In [ ]:
# ── Cell 4 — is it actually working? ──────────────────────────────────────
# Starting is not the same as working. This checks each thing that breaks
# independently and names the one that failed. It lives in the repo, so it
# improves with a `git pull` — no notebook re-import.
!python3 kaggle_smoke_test.py


## Verify this session's two fixes

1. **nvenc-capable ffmpeg** (`kaggle_bootstrap.sh`) — Kaggle's preinstalled
   ffmpeg has no `--enable-nvenc`, so `FFMPEG_ENCODER=nvenc` was silently
   falling back to libx264 (CPU) even though the GPU config asked for
   hardware encode. Cell 3 above installs a static nvenc build when needed;
   the cell below confirms it actually landed.
2. **Phase 0 eval harness** (`eval/`) — pure-Python metrics (crop motion
   within a shot, speaker-on-screen accuracy, sentence-boundary
   completeness) wired to a real `reframe_v2.render()` call. Needs a job's
   `metadata.json` (from a job already run through the dashboard, or the
   optional render-test cell below) to have a transcript to measure against.


In [ ]:
# ── nvenc check ─────────────────────────────────────────────────────────
import subprocess
enc = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"],
                     capture_output=True, text=True).stdout
if "h264_nvenc" in enc:
    print("OK   h264_nvenc is available to ffmpeg")
else:
    print("MISS h264_nvenc NOT available — check the 'nvenc-capable ffmpeg'")
    print("     section in cell 3's output above for why the install failed")

# The line the actual pipeline prints when it picks an encoder for a render —
# the real proof, not just "is the binary capable."
log = "/tmp/openshorts-logs/backend.log"
try:
    with open(log) as f:
        lines = [l for l in f if "[Encoder] video encoder:" in l]
    print(f"\nEncoder lines seen in {log} so far:")
    print("".join(lines[-5:]) if lines else "  (none yet — render a clip first, "
          "e.g. the optional render-test cell below)")
except FileNotFoundError:
    print(f"\n{log} not found yet — start the backend (cell 3) first")

# ── Phase 0 eval harness ────────────────────────────────────────────────
# Fill these in once you have a job's metadata.json (Job History in the
# dashboard, or /kaggle/working/openshorts/output/<job_id>/*_metadata.json).
SOURCE_VIDEO = ""   # e.g. "/kaggle/working/test.mp4"
METADATA_JSON = ""  # e.g. "/kaggle/working/openshorts/output/<job>/<base>_metadata.json"
SPAN = "0-20"       # "start-end" in seconds

import os
if SOURCE_VIDEO and METADATA_JSON and os.path.exists(SOURCE_VIDEO) and os.path.exists(METADATA_JSON):
    os.environ["REFRAME_DUMP_PATH"] = "/tmp/eval_dump"
    get_ipython().system(
        f'python3 eval/run.py "{SOURCE_VIDEO}" --span {SPAN} --metadata "{METADATA_JSON}"'
    )
else:
    print("Set SOURCE_VIDEO and METADATA_JSON above (after running a job) "
          "and re-run this cell to get real baseline numbers.")


## Optional — end-to-end render test (~1-2 min on GPU)

The real proof: reframe a short clip and read the framing telemetry. Uses a
source you upload yourself, so it works even if cookies are dead.


In [ ]:
# Point SRC at any short landscape video with speech (upload one to
# /kaggle/working, or attach a Kaggle Dataset and use /kaggle/input/...).
SRC = "/kaggle/working/test.mp4"

import os, time
if not os.path.exists(SRC):
    print(f"put a video at {SRC} first (or edit SRC)")
else:
    os.environ.setdefault("USE_ASD", "1")
    import reframe_v2 as r
    t0 = time.time()
    # No transcript here, so diarization is unavailable and LR-ASD carries the
    # speaker identification alone — a deliberately harder case than production.
    r.render(SRC, "/kaggle/working/test_vertical.mp4", 0.75)
    print(f"\nrendered in {time.time()-t0:.1f}s -> /kaggle/working/test_vertical.mp4")
    print("Look at the '🎯 Framing evidence' line above:")
    print("  lip-sync high + size ~0%  = working as intended")
    print("  size high                 = the speaker signal is not reaching the camera")

## Optional — keep the session alive

Jupyter runs **one cell at a time**, so a blocking `while True:` loop locks the
notebook. This runs the heartbeat on a background thread, so the cell returns
immediately.

You do not need it while actively clicking around; it matters when you walk
away mid-render or use *Save & Run All*.

**Stopping any cell is safe** — the backend and tunnel run under `nohup`.

**Clips survive the session only if HF storage is configured** (`HF_TOKEN` +
`HF_STORAGE_REPO` in cell 1). They upload as each clip finishes, and History
serves them from HF after `/kaggle/working` is wiped.


In [ ]:
import threading, time

def _heartbeat():
    # Touches a file rather than printing: notebook output from a background
    # thread interleaves with whatever cell you are running next, which makes
    # the notebook unreadable.
    while True:
        with open('/kaggle/working/.heartbeat', 'w') as f:
            f.write(str(time.time()))
        time.sleep(60)

if not any(t.name == 'openshorts-heartbeat' for t in threading.enumerate()):
    threading.Thread(target=_heartbeat, name='openshorts-heartbeat',
                     daemon=True).start()
    print('heartbeat started on a background thread — this cell is free')
else:
    print('heartbeat already running')
